In [1]:
import duckdb
con = duckdb.connect()
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of analysis + time window

**One row = one (report_date, client_hash_id, content_hash_id) combination**
— a single day's search-and-site performance snapshot for one piece of
content, for one client, from `fact_content_daily_performance`.

**Time window:** working/dev month = `month=2026-03` (mid-panel, per the
assignment's own warning). `2026-06` (the `_sample` file) is treated as a
sealed test month — used only to check query mechanics, never to develop
label logic, since it's the natural outcome window for any past→future
label.

In [2]:
print("Working month:", "2026-03", "| sealed test month:", "2026-06 (sample file, mechanics only)")

Working month: 2026-03 | sealed test month: 2026-06 (sample file, mechanics only)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

**Features (knowable at decision time, first half of the month only):**
- `gsc_avg_position` (first-half average)
- `gsc_impressions` (first-half sum)
- `gsc_clicks` (first-half sum, or CTR derived from it)
- `word_count` (from `dim_content` — static, known well before the month)
- content age at month start (`report_date` minus `content_created_date`,
  from `dim_content`)

**Label / proxy:** `declining_flag` — 1 if a content item's average
`gsc_avg_position` in the second half of the month is worse (numerically
higher) than in the first half, else 0. A within-month proxy, since a
single month's slice can't yet look at true forward months.

**Context (identifiers, not features):** `client_hash_id`,
`content_hash_id`, `report_date`, `month`.

**Deliberately excluded:** all `ga4_*` fields (pageviews, sessions,
engagement) and the `ai_*` / `sessions_ai` breakdown. This lane is about
*search ranking* signals specifically — GSC position, impressions, clicks
— not on-site engagement or AI-referral traffic, which belong to a
different lane's question.

In [3]:
fields = {
    "feature": ["gsc_avg_position", "gsc_impressions", "gsc_clicks", "word_count", "content_age_days"],
    "label_proxy": ["declining_flag (derived: 2nd-half avg_position worse than 1st-half)"],
    "context": ["client_hash_id", "content_hash_id", "report_date", "month"],
    "excluded": ["ga4_* fields", "ai_* / sessions_ai fields"],
}
for bucket, cols in fields.items():
    print(f"{bucket}: {cols}")

feature: ['gsc_avg_position', 'gsc_impressions', 'gsc_clicks', 'word_count', 'content_age_days']
label_proxy: ['declining_flag (derived: 2nd-half avg_position worse than 1st-half)']
context: ['client_hash_id', 'content_hash_id', 'report_date', 'month']
excluded: ['ga4_* fields', 'ai_* / sessions_ai fields']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Five features

1. **`first_half_avg_position`** — knowable at the decision moment because
   it's built only from days 1-15 of March; nothing from days 16-31 (the
   label window) touches it.
2. **`first_half_impressions`** — same reason: summed only over the first
   half of the month, before the outcome period exists.
3. **`first_half_clicks`** — same: first-half-only aggregate.
4. **`word_count`** — knowable because it's a static property of the
   content from `dim_content`, set whenever the page was last edited,
   long before this month's decision point.
5. **`content_age_days`** — knowable because it's just the gap between
   `content_created_date` and the start of the decision month — a fact
   about the past, not the future.

### The trap

Adding `leaky_second_half_position` — literally the same column the label
(`declining_flag`) is derived from — pushed the AUC from **0.674 to
1.0**, a jump of **0.326**. That's not a better model; it's the model
being handed the answer. A perfect score here is a red flag, not a win,
because in production the model would never have Mrach 16-31st's
actual average position available at the moment it needs to make a
prediction. The leaky column was deleted immediately after confirming the
jump, and the honest number (0.674) is the one that stays.

In [4]:
import duckdb
con = duckdb.connect()
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

fact_path = f"{rel}/fact_content_daily_performance/**/*.parquet"

# --- Query 1: grain check ---
print("Query 1 — grain check (should be 0 duplicate groups):")
print(con.sql(f"""
    SELECT COUNT(*) AS duplicate_groups
    FROM (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
        FROM read_parquet('{fact_path}', hive_partitioning=1)
        WHERE month = '2026-03'
        GROUP BY 1,2,3
        HAVING COUNT(*) > 1
    )
""").df())

# --- Query 2: row count + date span ---
print("\nQuery 2 — row count and date span for month=2026-03:")
print(con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM read_parquet('{fact_path}', hive_partitioning=1)
    WHERE month = '2026-03'
""").df())

# --- Query 3: availability, filtered with IS TRUE ---
print("\nQuery 3 — availability check:")
print(con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        ROUND(100.0 * COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) / COUNT(*), 1) AS pct_available
    FROM read_parquet('{fact_path}', hive_partitioning=1)
    WHERE month = '2026-03'
""").df())

Query 1 — grain check (should be 0 duplicate groups):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   duplicate_groups
0                 0

Query 2 — row count and date span for month=2026-03:
   total_rows   min_date   max_date
0     9841378 2026-03-01 2026-03-31

Query 3 — availability check:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  gsc_available_rows  pct_available
0     9841378             3611061           36.7


In [5]:
# Build first-half features (days 1-15) and second-half label (days 16-31),
# from the same month=2026-03 — nothing here reaches past March.

feature_query = f"""
WITH daily AS (
    SELECT *
    FROM read_parquet('{fact_path}', hive_partitioning=1)
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
),
first_half AS (
    SELECT client_hash_id, content_hash_id,
           AVG(gsc_avg_position) AS first_half_avg_position,
           SUM(gsc_impressions)  AS first_half_impressions,
           SUM(gsc_clicks)       AS first_half_clicks
    FROM daily
    WHERE report_date <= DATE '2026-03-15'
    GROUP BY 1,2
),
second_half AS (
    SELECT client_hash_id, content_hash_id,
           AVG(gsc_avg_position) AS second_half_avg_position
    FROM daily
    WHERE report_date > DATE '2026-03-15'
    GROUP BY 1,2
)
SELECT
    f.client_hash_id, f.content_hash_id,
    f.first_half_avg_position, f.first_half_impressions, f.first_half_clicks,
    c.word_count,
    DATE_DIFF('day', c.content_created_date, DATE '2026-03-01') AS content_age_days,
    s.second_half_avg_position,
    CASE WHEN s.second_half_avg_position > f.first_half_avg_position THEN 1 ELSE 0 END AS declining_flag
FROM first_half f
JOIN second_half s USING (client_hash_id, content_hash_id)
JOIN read_parquet('{rel}/dim_content.parquet') c USING (client_hash_id, content_hash_id)
WHERE f.first_half_avg_position IS NOT NULL AND s.second_half_avg_position IS NOT NULL
"""

df = con.sql(feature_query).df()
print("Feature frame shape:", df.shape)
df.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (141467, 9)


,client_hash_id,content_hash_id,first_half_avg_position,first_half_impressions,first_half_clicks,word_count,content_age_days,second_half_avg_position,declining_flag
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,12.639599,119.0,1.0,3168,145,15.525721,1
1,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,8.094074,72.0,0.0,3211,145,8.847778,1
2,client_0797ff3a1fc9a6a5,content_1207efddce873942,12.587226,283.0,0.0,3465,145,16.990390,1
3,client_0797ff3a1fc9a6a5,content_14df4b67b008d942,11.500000,9.0,0.0,3201,145,7.873457,0
4,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,12.282875,66.0,0.0,3149,145,12.309424,1
5,client_0797ff3a1fc9a6a5,content_1fea2f270f3c1350,3.500000,2.0,0.0,4071,154,4.500000,1
6,client_0797ff3a1fc9a6a5,content_27f8100281413b37,8.083333,11.0,0.0,4016,145,9.738726,1
7,client_0797ff3a1fc9a6a5,content_2f719399052f18fc,21.250000,8.0,0.0,4821,145,26.798611,1
8,client_0797ff3a1fc9a6a5,content_37952b007ab057b3,11.741716,440.0,2.0,3804,145,14.064750,1
9,client_0797ff3a1fc9a6a5,content_3cf5722aa6fd767f,9.366667,14.0,0.0,3698,145,8.824074,0


In [6]:
# Quick honest score using only the 5 real features
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import numpy as np

feature_cols = ["first_half_avg_position", "first_half_impressions",
                 "first_half_clicks", "word_count", "content_age_days"]

model_df = df.dropna(subset=feature_cols + ["declining_flag"])
X = model_df[feature_cols]
y = model_df["declining_flag"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])
print("Honest AUC (5 real features):", round(honest_auc, 3))

Honest AUC (5 real features): 0.677


In [7]:
# THE TRAP: add ONE label-derived column on purpose
model_df_leak = model_df.copy()
model_df_leak["leaky_second_half_position"] = model_df_leak["second_half_avg_position"]  # <- this IS the label's source

leak_cols = feature_cols + ["leaky_second_half_position"]
X_leak = model_df_leak[leak_cols]
y_leak = model_df_leak["declining_flag"]

X_train, X_test, y_train, y_test = train_test_split(X_leak, y_leak, test_size=0.3, random_state=42)
clf_leak = LogisticRegression(max_iter=1000).fit(X_train, y_train)
leaky_auc = roc_auc_score(y_test, clf_leak.predict_proba(X_test)[:, 1])
print("Leaky AUC (with label-derived column):", round(leaky_auc, 3))
print("Jump:", round(leaky_auc - honest_auc, 3))

Leaky AUC (with label-derived column): 1.0
Jump: 0.323


In [8]:
# Delete the leak, keep the honest number
del model_df_leak["leaky_second_half_position"]
print("Leak removed. Keeping the honest AUC:", round(honest_auc, 3))

Leak removed. Keeping the honest AUC: 0.677


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Only **36.7%** of rows in `month=2026-03` have `gsc_data_available IS
TRUE` — meaning for roughly two-thirds of client-content-day
combinations, this lane's core signal (search position, impressions,
clicks) simply doesn't exist for that row. This matches the warehouse's
own documented "unbalanced panel" design — per-client GSC/GA4 history
depth differs (`dim_clients.gsc_data_start` varies by client) — so any
model trained on this slice is implicitly a model of *clients with GSC
history in March 2026*, not of all clients equally.

Separately, **5.1% of rows (7,189 of 141,467)** in the feature frame have
a negative `content_age_days` — content marked as "created" a few days
*after* the report month began (e.g. -3 days). This isn't a rounding
artifact or a one-off row; it's a real, consistent pattern affecting one
in twenty rows, likely a timestamp/backdating quirk in the source system
rather than an actual negative age. This needs a sanity filter
(`content_age_days >= 0`, or investigating why so many rows share this
pattern) before it's trusted as a feature in any real model — not just a
footnote to note and move past.

In [9]:
# Claim 1: availability split (reprint from Section 3 for direct evidence here)
print("GSC availability in month=2026-03:")
print(con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        ROUND(100.0 * COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) / COUNT(*), 1) AS pct_available
    FROM read_parquet('{fact_path}', hive_partitioning=1)
    WHERE month = '2026-03'
""").df())

# Claim 2: negative content_age_days check
negative_age_count = (df["content_age_days"] < 0).sum()
print(f"\nRows with negative content_age_days: {negative_age_count} out of {len(df)}")
print(df[df['content_age_days'] < 0][['content_hash_id', 'content_age_days']].head())

GSC availability in month=2026-03:
   total_rows  gsc_available_rows  pct_available
0     9841378             3611061           36.7

Rows with negative content_age_days: 7189 out of 141467
               content_hash_id  content_age_days
710   content_0a49f276c4569b27                -3
1041  content_0f6f862dc21e0716                -3
1063  content_0fee67b6494c81e6                -3
1302  content_136143b657ed39a2                -3
1397  content_14bf469440d2787f                -3


## Self-check

Before you submit, confirm each line honestly:

- [ ✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅ ] No client names, URLs, or private queries anywhere
- [✅ ] My claims use careful words: observed, measured, directional, decision-support
- [✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.